In [8]:
%load_ext autoreload
%autoreload 2
from src.data_utils import DataUtils
import yaml
import pandas as pd
from src.utils import my_device
from src.next_token_dataset import NextTokenDataset
from transformers import BertTokenizerFast
from torch.utils.data import DataLoader
import torch
from src.lstm_model import LstmModel
from transformers import pipeline
from transformers import AutoTokenizer
from src.transformer import transformer_make_prediction
import evaluate

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# Загрузка конфига
with open("configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

In [ ]:
# Создание csv файлов, если есть только исходники
# DataUtils.samples_create(config['dataset'])

In [3]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

df_train = pd.read_csv(config['dataset']['path'] + '/train.csv')
df_val = pd.read_csv(config['dataset']['path'] + '/val.csv')

# train_dataset = NextTokenDataset(df_train, tokenizer, max_len=16)
# val_dataset = NextTokenDataset(df_val, tokenizer, max_len=16)

train_dataset = NextTokenDataset(df_train["text"].tolist(), tokenizer, max_len=16)
val_dataset = NextTokenDataset(df_val["text"].tolist(), tokenizer, max_len=16)

# num_workers=0 - чтобы не было дедлоков
# The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=0)


In [4]:
for batch in train_loader:
    print(batch["input_ids"].shape)
    break

torch.Size([256, 15])


In [5]:
from src.eval_transformer_pipeline import eval_transformer_pipeline
model = LstmModel(vocab_size=tokenizer.vocab_size, hidden_dim=128).to(my_device())

eval_transformer_pipeline(
    config=config,
    model=model,
    tokenizer=tokenizer,
    train_loader=train_loader,
    val_loader=val_loader
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Epoch 1/5 | Train Loss: 5.3885 | Val Loss: 4.9642 | Val Acc: 0.2878 | rog 0.0898
Epoch 2/5 | Train Loss: 4.8453 | Val Loss: 4.8204 | Val Acc: 0.2982 | rog 0.1031


Epoch 3/5 | Train Loss: 4.7550 | Val Loss: 4.7219 | Val Acc: 0.3073 | rog 0.1135
Epoch 4/5 | Train Loss: 4.7099 | Val Loss: 4.6936 | Val Acc: 0.3098 | rog 0.1176
Epoch 5/5 | Train Loss: 4.6927 | Val Loss: 4.6766 | Val Acc: 0.3113 | rog 0.1146


In [6]:
model.eval()
for i in range(10):
    text = df_train.iloc[i]
    if isinstance(text, (pd.Series, dict)):
        text = text[0]  # если DataFrame с одной колонкой
    prompt = text.split()[: len(text.split()) * 3 // 4]
    prompt_str = " ".join(prompt)

    # Токенизируем "частичный" текст
    input_ids = tokenizer.encode(prompt_str, return_tensors="pt", truncation=True, max_length=32)

    # Генерируем дополнение
    generated_ids = model.generate(input_ids, max_new_tokens=10)
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    print(f"🟢 Original: {text}")
    print(f"🔹 Prompt: {prompt_str}")
    print(f"🔸 Generated: {generated_text}\n{'-'*80}")


/var/folders/2p/1g0tbznd1zg9118y89qrclsc0000gn/T/ipykernel_17719/1940004646.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  text = text[0]  # если DataFrame с одной колонкой


🟢 Original: hanrudman aah lovely say hello to the ntw peeps swam in the pool sea paddling only but tonight ill swim in the sea for sure
🔹 Prompt: hanrudman aah lovely say hello to the ntw peeps swam in the pool sea paddling only but tonight
🔸 Generated: hanrudman aah lovely say hello to the ntw peeps swam in the pool sea paddling only but tonight
--------------------------------------------------------------------------------
🟢 Original: cjcroll why are you sad
🔹 Prompt: cjcroll why are
🔸 Generated: cjcroll why are
--------------------------------------------------------------------------------
🟢 Original: beyonces music is amazinggg lt3 love her ugh im so tired about to go to bed hope everyone has a good day tomorrow night sillys
🔹 Prompt: beyonces music is amazinggg lt3 love her ugh im so tired about to go to bed hope everyone
🔸 Generated: beyonces music is amazinggg lt3 love her ugh im so tired about to go to bed hope everyone
--------------------------------------------------------

In [12]:

generator = pipeline("text-generation", model="distilgpt2")
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")


Device set to use mps:0


In [ ]:
# Тестирование моделей
df_test = pd.read_csv(config['dataset']['path'] + '/test.csv')

rouge = evaluate.load("rouge")

for i in range(10):
    text = df_test.iloc[i]["text"]
    words = text.split()
    split_point = len(words) * 3 // 4
    prompt = words[:split_point]
    prompt_str = " ".join(prompt)
    reference = " ".join(words[split_point:]) # Часть для сравнения с генерацией

    print('PROMPT: ' + prompt_str)
    # LSTM
    model.eval()
    prompt_tokens = tokenizer.encode(prompt_str, return_tensors='pt').to(my_device())
    with torch.no_grad():
            lstm_generated_tokens = model.generate(
                prompt_tokens, 
                max_new_tokens=20,
            )
    lstm_text = tokenizer.decode(lstm_generated_tokens[0], skip_special_tokens=True)
    lstm_rouge_scores = rouge.compute(
            predictions=[lstm_text[len(prompt_str):].strip()], 
            references=[reference]
    )
    print(f"LSTM result: {lstm_text}")
    print(f"ROGUE-1 {lstm_rouge_scores['rouge1']:.4f}")
    print(f"ROGUE-2 {lstm_rouge_scores['rouge2']:.4f}")
    print(f"ROGUE-L {lstm_rouge_scores['rougeL']:.4f}")

    trns_output = generator(text, max_new_tokens=20, do_sample=True, top_k=50, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
    trns_text = trns_output[0]["generated_text"]  # Извлекаем текст из первого результата
    trns_gen_only = trns_text[len(prompt_str):].strip()

    trns_rouge_scores = rouge.compute(
            predictions=[trns_gen_only], 
            references=[reference]
        )

    print('TNSF result: '+ trns_text.replace("\n", " ").strip())
    print(f"ROGUE-1 {trns_rouge_scores['rouge1']:.4f}")
    print(f"ROGUE-2 {trns_rouge_scores['rouge2']:.4f}")
    print(f"ROGUE-L {trns_rouge_scores['rougeL']:.4f}")

    # Модель distilgpt2
    transformer_make_prediction(generator,tokenizer, text)
    print('-'*80)




PROMPT: jonnyboyslim hah that would be a funny sight sleep time mark is makin me go to town v early
LSTM result: jonnyboyslim hah that would be a funny sight sleep time mark is makin me go to town v early�������������!!!!!!!
ROGUE-1 0.0000
ROGUE-2 0.0000
ROGUE-L 0.0000
TNSF result: jonnyboyslim hah that would be a funny sight sleep time mark is makin me go to town v early tomorrow for last minute shoppin night xh what a great day of a night and I have to do it with the best of my ability
ROGUE-1 0.3636
ROGUE-2 0.3226
TNSF result: jonnyboyslim hah that would be a funny sight sleep time mark is makin me go to town v early tomorrow for last minute shoppin night xh.   I'm sorry to hear that you're not going to be able to sleep
--------------------------------------------------------------------------------
PROMPT: christinemc0828 yes i will admit i was pleasantly surprised with the quality of
LSTM result: christinemc0828 yes i will admit i was pleasantly surprised with the quality of������